# Olist Brazilian E-Commerce — Data Analysis Hackathon

## Objective

This project analyzes the Olist Brazilian E-Commerce Public Dataset to evaluate marketplace performance, customer satisfaction, delivery performance, seller and geographic patterns, product categories, payment behavior, and the factors associated with low customer review scores.

## Key Business Questions

1. How has marketplace performance changed over time?
2. How does delivery performance relate to customer satisfaction?
3. Which sellers and geographic regions show performance differences?
4. Which product categories perform better or worse?
5. How does payment behavior relate to order value and customer experience?
6. What factors are most strongly associated with low review scores?

## Analytical Approach

The analysis uses exploratory data analysis, aggregation, correlation analysis, segmentation, and comparative analysis. The findings describe observed associations in the data and do not claim causal relationships.

# 1. Data Loading and Initial Setup

The analysis uses the nine tables supplied in the Olist workbook. The tables are loaded into pandas DataFrames for subsequent cleaning, integration, and analysis.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings("ignore")
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)

print("Libraries imported successfully!")

In [ ]:


file_path = "/content/Document from singhlink4.xlsx"

orders = pd.read_excel(file_path, sheet_name="orders")
order_items = pd.read_excel(file_path, sheet_name="order_items")
order_payments = pd.read_excel(file_path, sheet_name="order_payments")
order_reviews = pd.read_excel(file_path, sheet_name="order_reviews")
customers = pd.read_excel(file_path, sheet_name="customers")
products = pd.read_excel(file_path, sheet_name="products")
sellers = pd.read_excel(file_path, sheet_name="sellers")
geolocation = pd.read_excel(file_path, sheet_name="geolocation")
category_translation = pd.read_excel(file_path, sheet_name="category_translation")

print("All datasets loaded successfully!")

In [ ]:
# Dataset overview
datasets = {
    "Orders": orders,
    "Order Items": order_items,
    "Order Payments": order_payments,
    "Order Reviews": order_reviews,
    "Customers": customers,
    "Products": products,
    "Sellers": sellers,
    "Geolocation": geolocation,
    "Category Translation": category_translation
}

for name, df in datasets.items():
    print(f"{name}: {df.shape[0]:,} rows × {df.shape[1]} columns")

# 2. Data Understanding and Dataset Overview

The Olist Brazilian E-Commerce dataset consists of multiple interconnected tables covering orders, customers, products, sellers, payments, reviews, geographic information, and product category translations.

Understanding the structure and relationships between these datasets is essential before performing the analysis.

In [ ]:
# Display column names for each dataset

for name, df in datasets.items():
    print(f"\n{name}")
    print("-" * len(name))
    print(list(df.columns))

### Key observation

The `orders` table is the main order-level table. Other tables provide item, payment, review, customer, product, seller, geographic, and category information that can be aggregated and joined without changing the intended order-level grain.

# 3. Dataset Relationships

The main relationships used in this analysis are:

* `orders` → `order_items` through `order_id`
* `orders` → `order_payments` through `order_id`
* `orders` → `order_reviews` through `order_id`
* `orders` → `customers` through `customer_id`
* `order_items` → `products` through `product_id`
* `order_items` → `sellers` through `seller_id`
* `products` → `category_translation` through `product_category_name`

These relationships allow order-level, customer-level, product-level, seller-level, payment, review, and geographic information to be combined for analysis.

In [ ]:
# Check key relationships between datasets

print("Orders:", orders["order_id"].nunique())
print("Order Items:", order_items["order_id"].nunique())
print("Order Payments:", order_payments["order_id"].nunique())
print("Order Reviews:", order_reviews["order_id"].nunique())
print("Customers:", customers["customer_id"].nunique())
print("Products:", products["product_id"].nunique())
print("Sellers:", sellers["seller_id"].nunique())

# 4. Data Quality Assessment

Before performing detailed analysis, the datasets are checked for missing values, duplicate records, and order-status distribution.

Missing values are not automatically removed because some missing delivery timestamps are expected for orders that were not delivered. These values are handled only within analyses where the relevant timestamp is required.

In [ ]:
# Missing values in Orders dataset

missing_orders = (
    orders.isna()
    .sum()
    .sort_values(ascending=False)
)

missing_orders[missing_orders > 0]

In [ ]:
# Duplicate checks

print("Duplicate rows:", orders.duplicated().sum())
print("Duplicate order IDs:", orders["order_id"].duplicated().sum())

In [ ]:
# Order status distribution

order_status_summary = (
    orders["order_status"]
    .value_counts()
    .reset_index()
)

order_status_summary.columns = ["Order Status", "Number of Orders"]

order_status_summary

In [ ]:
plt.figure(figsize=(9, 5))

orders["order_status"].value_counts().plot(kind="bar")

plt.title("Order Status Distribution")
plt.xlabel("Order Status")
plt.ylabel("Number of Orders")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

# 5. Data Preparation and Feature Engineering

Additional features are created from the original order and delivery timestamps to support time-based and delivery-performance analysis.

The main engineered features include purchase year, purchase month, purchase year-month, delivery duration, delivery delay, and delivery performance.

For delivery performance, only delivered orders with valid delivery dates are classified as `On Time` or `Late`. Non-delivered orders remain missing rather than being incorrectly classified as late.

In [ ]:
date_columns = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date"
]

for col in date_columns:
    orders[col] = pd.to_datetime(orders[col], errors="coerce")

orders[date_columns].dtypes

In [ ]:
orders["purchase_date"] = orders["order_purchase_timestamp"].dt.date

orders["purchase_year"] = orders["order_purchase_timestamp"].dt.year

orders["purchase_month"] = orders["order_purchase_timestamp"].dt.month

orders["purchase_month_name"] = orders["order_purchase_timestamp"].dt.month_name()

orders["purchase_year_month"] = (
    orders["order_purchase_timestamp"]
    .dt.to_period("M")
    .astype(str)
)

In [ ]:
orders["delivery_days"] = (
    orders["order_delivered_customer_date"]
    - orders["order_purchase_timestamp"]
).dt.total_seconds() / (24 * 60 * 60)

In [ ]:
orders["delivery_delay_days"] = (
    orders["order_delivered_customer_date"]
    - orders["order_estimated_delivery_date"]
).dt.total_seconds() / (24 * 60 * 60)

In [ ]:
order_item_summary = (
    order_items
    .groupby("order_id")
    .agg(
        total_items=("order_item_id", "count"),
        total_price=("price", "sum"),
        total_freight=("freight_value", "sum"),
        seller_count=("seller_id", "nunique"),
        product_count=("product_id", "nunique")
    )
    .reset_index()
)

order_item_summary.head()

In [ ]:
payment_summary = (
    order_payments
    .groupby("order_id")
    .agg(
        total_payment=("payment_value", "sum"),
        payment_count=("payment_sequential", "count"),
        payment_types=("payment_type", "nunique"),
        max_installments=("payment_installments", "max")
    )
    .reset_index()
)

payment_summary.head()

In [ ]:
payment_type_summary = (
    order_payments
    .groupby("order_id")["payment_type"]
    .agg(lambda x: ", ".join(sorted(x.dropna().unique())))
    .reset_index()
)

payment_type_summary.columns = ["order_id", "payment_types_used"]

payment_type_summary.head()

In [ ]:
master = orders.merge(
    order_item_summary,
    on="order_id",
    how="left"
)

master = master.merge(
    payment_summary,
    on="order_id",
    how="left"
)

master = master.merge(
    payment_type_summary,
    on="order_id",
    how="left"
)

master.shape

In [ ]:
review_summary = (
    order_reviews
    .groupby("order_id")
    .agg(
        review_score=("review_score", "mean")
    )
    .reset_index()
)

review_summary.head()

In [ ]:
master = master.merge(
    review_summary,
    on="order_id",
    how="left"
)

In [ ]:
customer_info = customers[
    [
        "customer_id",
        "customer_unique_id",
        "customer_zip_code_prefix",
        "customer_city",
        "customer_state"
    ]
]

master = master.merge(
    customer_info,
    on="customer_id",
    how="left"
)

In [ ]:
product_category = products.merge(
    category_translation,
    on="product_category_name",
    how="left"
)

product_category.head()

In [ ]:
items_with_category = order_items.merge(
    product_category[
        [
            "product_id",
            "product_category_name",
            "product_category_name_english"
        ]
    ],
    on="product_id",
    how="left"
)

items_with_category.head()

In [ ]:
category_summary = (
    items_with_category
    .groupby("order_id")
    .agg(
        category_count=(
            "product_category_name_english",
            "nunique"
        ),
        categories=(
            "product_category_name_english",
            lambda x: ", ".join(
                sorted(x.dropna().unique())
            )
        )
    )
    .reset_index()
)

category_summary.head()

In [ ]:
master = master.merge(
    category_summary,
    on="order_id",
    how="left"
)

print("New Master Shape:", master.shape)

In [ ]:
# Correct delivery performance classification

orders["delivery_performance"] = pd.NA

mask = (
    orders["order_status"].eq("delivered")
    & orders["delivery_delay_days"].notna()
)

orders.loc[mask, "delivery_performance"] = np.where(
    orders.loc[mask, "delivery_delay_days"] <= 0,
    "On Time",
    "Late"
)

# Update master dataset
master["delivery_performance"] = orders["delivery_performance"]

# Check result
print(master["delivery_performance"].value_counts(dropna=False))

### Preparation note

The master dataset remains at one row per order. Item-level and payment-level tables are aggregated to `order_id` before joining so that multi-item and multi-payment orders do not create unintended row multiplication.

# 6. Marketplace Performance Over Time

This section analyzes marketplace performance over time using monthly order volume, revenue, and average customer review score.

The objective is to identify growth patterns, changes in customer experience, and periods of stronger or weaker marketplace performance.

In [ ]:
monthly_kpi = (
    master.groupby("purchase_year_month")
    .agg(
        order_volume=("order_id", "nunique"),
        revenue=("total_payment", "sum"),
        avg_review_score=("review_score", "mean")
    )
    .reset_index()
)

monthly_kpi.head(10)

In [ ]:
plt.figure(figsize=(12, 5))

plt.plot(
    monthly_kpi["purchase_year_month"],
    monthly_kpi["order_volume"],
    marker="o"
)

plt.title("Monthly Order Volume")
plt.xlabel("Month")
plt.ylabel("Number of Orders")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(12, 5))

plt.plot(
    monthly_kpi["purchase_year_month"],
    monthly_kpi["revenue"],
    marker="o"
)

plt.title("Monthly Revenue Trend")
plt.xlabel("Month")
plt.ylabel("Revenue")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(12, 5))

plt.plot(
    monthly_kpi["purchase_year_month"],
    monthly_kpi["avg_review_score"],
    marker="o"
)

plt.title("Monthly Average Review Score")
plt.xlabel("Month")
plt.ylabel("Average Review Score")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

### Interpretation

The monthly analysis provides a view of marketplace growth and customer experience over the observed period. Order volume and revenue show the evolution of marketplace activity, while average review score provides a complementary customer-satisfaction measure.

# 7. Delivery Performance and Customer Satisfaction

Delivery performance is analyzed against customer review scores to understand whether delayed deliveries are associated with lower customer satisfaction.

The analysis compares on-time and late deliveries and then examines how review scores change as delivery delays become more severe.

In [ ]:
delivery_review = (
    master[master["delivery_performance"].notna()]
    .groupby("delivery_performance")
    .agg(
        orders=("order_id", "nunique"),
        avg_review_score=("review_score", "mean")
    )
    .reset_index()
)

delivery_review

In [ ]:
plt.figure(figsize=(8, 5))

plt.bar(
    delivery_review["delivery_performance"],
    delivery_review["avg_review_score"]
)

plt.title("Delivery Performance vs Average Review Score")
plt.xlabel("Delivery Performance")
plt.ylabel("Average Review Score")
plt.ylim(0, 5)

plt.tight_layout()
plt.show()

In [ ]:
master["delay_bucket"] = pd.cut(
    master["delivery_delay_days"],
    bins=[-np.inf, -7, 0, 3, 7, 14, np.inf],
    labels=[
        "7+ Days Early",
        "On Time",
        "1-3 Days Late",
        "4-7 Days Late",
        "8-14 Days Late",
        "15+ Days Late"
    ]
)

delay_review = (
    master[master["delay_bucket"].notna()]
    .groupby("delay_bucket", observed=True)
    .agg(
        orders=("order_id", "nunique"),
        avg_review_score=("review_score", "mean")
    )
    .reset_index()
)

delay_review

In [ ]:
plt.figure(figsize=(10, 5))

plt.plot(
    delay_review["delay_bucket"].astype(str),
    delay_review["avg_review_score"],
    marker="o"
)

plt.title("Delivery Delay Severity vs Average Review Score")
plt.xlabel("Delivery Delay")
plt.ylabel("Average Review Score")
plt.ylim(0, 5)
plt.xticks(rotation=30)

plt.tight_layout()
plt.show()

### Key finding

Delivery performance shows a strong relationship with customer satisfaction. Low-review rates rise sharply as delivery delays become more severe, making delivery reliability the strongest observed factor associated with dissatisfaction.

# 8. Seller and Geographic Analysis

Geographic differences in delivery performance and customer satisfaction are analyzed at the customer-state level.

States with fewer than 100 delivered orders are excluded from the detailed comparison to reduce the influence of very small samples.

In [ ]:
state_delivery = (
    master[master["delivery_performance"].notna()]
    .groupby("customer_state")
    .agg(
        orders=("order_id", "nunique"),
        avg_delivery_days=("delivery_days", "mean"),
        avg_delay_days=("delivery_delay_days", "mean"),
        avg_review_score=("review_score", "mean")
    )
    .reset_index()
)

late_rate = (
    master[master["delivery_performance"].notna()]
    .assign(
        is_late=lambda x: x["delivery_performance"].eq("Late")
    )
    .groupby("customer_state")["is_late"]
    .mean()
    .mul(100)
    .reset_index(name="late_delivery_rate")
)

state_delivery = state_delivery.merge(
    late_rate,
    on="customer_state",
    how="left"
)

state_analysis = (
    state_delivery[state_delivery["orders"] >= 100]
    .sort_values("late_delivery_rate", ascending=False)
)

state_analysis.head(10)

In [ ]:
top_late_states = state_analysis.head(10)

plt.figure(figsize=(10, 5))

plt.bar(
    top_late_states["customer_state"],
    top_late_states["late_delivery_rate"]
)

plt.title("Top 10 States by Late Delivery Rate")
plt.xlabel("Customer State")
plt.ylabel("Late Delivery Rate (%)")

plt.tight_layout()
plt.show()

In [ ]:
correlation = state_analysis[
    ["late_delivery_rate", "avg_review_score"]
].corr().iloc[0, 1]

print(
    f"Correlation between Late Delivery Rate and Review Score: "
    f"{correlation:.3f}"
)

### Key finding

State-level analysis shows a strong negative association between late-delivery rate and average review score. The Pearson correlation is approximately **−0.874**, indicating that states with higher late-delivery rates tend to have lower average review scores. This is an association, not proof of causation.

# 9. Product Category Performance

Product categories are analyzed based on order volume, revenue, average product price, and customer review scores.

Categories with very small order volumes are treated cautiously when interpreting their performance.

In [ ]:
items_category = order_items.merge(
    product_category[
        ["product_id", "product_category_name_english"]
    ],
    on="product_id",
    how="left"
)

items_category = items_category.merge(
    master[["order_id", "review_score"]],
    on="order_id",
    how="left"
)

category_performance = (
    items_category
    .groupby("product_category_name_english")
    .agg(
        order_items=("order_id", "count"),
        unique_orders=("order_id", "nunique"),
        revenue=("price", "sum"),
        avg_price=("price", "mean"),
        avg_review_score=("review_score", "mean")
    )
    .reset_index()
)

category_performance.head(10)

In [ ]:
top_categories = (
    category_performance
    .sort_values("unique_orders", ascending=False)
    .head(10)
)

plt.figure(figsize=(10, 6))

plt.barh(
    top_categories["product_category_name_english"],
    top_categories["unique_orders"]
)

plt.title("Top 10 Product Categories by Order Volume")
plt.xlabel("Number of Orders")
plt.ylabel("Product Category")
plt.gca().invert_yaxis()

plt.tight_layout()
plt.show()

In [ ]:
category_review = (
    category_performance[
        category_performance["unique_orders"] >= 100
    ]
    .sort_values("avg_review_score")
)

category_review.head(10)

In [ ]:
lowest_categories = category_review.head(10)

plt.figure(figsize=(10, 6))

plt.barh(
    lowest_categories["product_category_name_english"],
    lowest_categories["avg_review_score"]
)

plt.title("Lowest-Rated Product Categories")
plt.xlabel("Average Review Score")
plt.ylabel("Product Category")
plt.xlim(0, 5)
plt.gca().invert_yaxis()

plt.tight_layout()
plt.show()

### Key finding

Product categories show meaningful differences in customer satisfaction. Several high-volume categories have review performance below the marketplace average, but category effects are weaker than the delivery-related signals identified in the root-cause analysis.

# 10. Payment Behavior Analysis

Payment behavior is analyzed using payment type, average order value, installment count, and customer review scores.

The objective is to understand whether different payment methods or installment patterns are associated with differences in order value and customer experience.

In [ ]:
payment_analysis = (
    master[master["total_payment"].notna()]
    .groupby("payment_types_used")
    .agg(
        orders=("order_id", "nunique"),
        avg_order_value=("total_payment", "mean"),
        avg_review_score=("review_score", "mean"),
        avg_installments=("max_installments", "mean")
    )
    .reset_index()
    .sort_values("orders", ascending=False)
)

payment_analysis

In [ ]:
payment_chart = payment_analysis[
    payment_analysis["orders"] >= 100
]

plt.figure(figsize=(10, 5))

plt.bar(
    payment_chart["payment_types_used"],
    payment_chart["avg_order_value"]
)

plt.title("Payment Type vs Average Order Value")
plt.xlabel("Payment Type")
plt.ylabel("Average Order Value")
plt.xticks(rotation=30)

plt.tight_layout()
plt.show()

In [ ]:
installment_analysis = (
    master[master["max_installments"].notna()]
    .groupby("max_installments")
    .agg(
        orders=("order_id", "nunique"),
        avg_order_value=("total_payment", "mean"),
        avg_review_score=("review_score", "mean")
    )
    .reset_index()
    .sort_values("max_installments")
)

installment_analysis

In [ ]:
installment_chart = installment_analysis[
    installment_analysis["orders"] >= 100
]

plt.figure(figsize=(10, 5))

plt.plot(
    installment_chart["max_installments"],
    installment_chart["avg_order_value"],
    marker="o"
)

plt.title("Installment Count vs Average Order Value")
plt.xlabel("Maximum Installments")
plt.ylabel("Average Order Value")
plt.xticks(installment_chart["max_installments"])

plt.tight_layout()
plt.show()

### Key finding

Payment behavior appears more strongly related to order value than to customer satisfaction. Credit cards dominate the marketplace, while higher installment counts generally correspond to higher order values. Review-score differences across major payment groups are limited.

# 11. Root Cause Analysis

This section identifies the factors most strongly associated with low customer review scores.

A low-review order is defined as an order receiving a review score of 1 or 2. Delivery reliability and delay severity are compared with other potential contributors to determine their relative importance.

The analysis identifies associations and potential contributors; it does not claim that any factor directly causes dissatisfaction.

In [ ]:
master["low_review"] = np.where(
    master["review_score"] <= 2,
    1,
    0
)

master["low_review"].value_counts()

In [ ]:
low_review_delivery = (
    master[master["delivery_performance"].notna()]
    .groupby("delivery_performance")
    .agg(
        orders=("order_id", "nunique"),
        low_review_orders=("low_review", "sum"),
        low_review_rate=("low_review", "mean")
    )
    .reset_index()
)

low_review_delivery["low_review_rate"] *= 100

low_review_delivery

In [ ]:
low_review_delay = (
    master[master["delay_bucket"].notna()]
    .groupby("delay_bucket", observed=True)
    .agg(
        orders=("order_id", "nunique"),
        low_review_orders=("low_review", "sum"),
        low_review_rate=("low_review", "mean"),
        avg_review_score=("review_score", "mean")
    )
    .reset_index()
)

low_review_delay["low_review_rate"] *= 100

low_review_delay

In [ ]:
plt.figure(figsize=(10, 5))

plt.plot(
    low_review_delay["delay_bucket"].astype(str),
    low_review_delay["low_review_rate"],
    marker="o"
)

plt.title("Delivery Delay Severity vs Low Review Rate")
plt.xlabel("Delivery Delay")
plt.ylabel("Low Review Rate (%)")
plt.xticks(rotation=30)

plt.tight_layout()
plt.show()

### Key finding

Late orders have a **54.56% low-review rate**, compared with **9.46% for on-time orders**. The low-review rate increases further with severe delays, reaching more than 78% for orders delayed by 8 days or more.

# 12. Additional Root Cause Factors

In addition to delivery performance, freight cost and product category are analyzed as potential contributors to customer dissatisfaction.

Freight cost is evaluated relative to product value using the freight-to-price ratio. Product categories are compared using order volume, average review score, and low-review rate.

These factors are interpreted as secondary contributors because their observed relationships with customer satisfaction are weaker than the delivery-related factors.

In [ ]:
master["freight_ratio"] = (
    master["total_freight"] / master["total_price"]
)

freight_data = master[
    master["freight_ratio"].notna()
    & master["review_score"].notna()
].copy()

freight_data["freight_group"] = pd.qcut(
    freight_data["freight_ratio"],
    q=5,
    duplicates="drop"
)

freight_analysis = (
    freight_data
    .groupby("freight_group", observed=True)
    .agg(
        orders=("order_id", "nunique"),
        avg_freight_ratio=("freight_ratio", "mean"),
        avg_review_score=("review_score", "mean"),
        low_review_rate=("low_review", "mean")
    )
    .reset_index()
)

freight_analysis["low_review_rate"] *= 100

freight_analysis

In [ ]:
plt.figure(figsize=(10, 5))

plt.plot(
    freight_analysis["freight_group"].astype(str),
    freight_analysis["avg_review_score"],
    marker="o"
)

plt.title("Freight-to-Price Ratio vs Average Review Score")
plt.xlabel("Freight-to-Price Ratio Group")
plt.ylabel("Average Review Score")
plt.ylim(0, 5)
plt.xticks(rotation=25)

plt.tight_layout()
plt.show()

In [ ]:
category_low_review = (
    items_category[
        items_category["product_category_name_english"].notna()
    ]
    .groupby("product_category_name_english")
    .agg(
        orders=("order_id", "nunique"),
        avg_review_score=("review_score", "mean"),
        low_review_rate=(
            "review_score",
            lambda x: (x <= 2).mean() * 100
        )
    )
    .reset_index()
)

category_low_review_filtered = (
    category_low_review[
        category_low_review["orders"] >= 100
    ]
    .sort_values("low_review_rate", ascending=False)
)

category_low_review_filtered.head(10)


In [ ]:
category_delivery = (
    items_category[
        ["order_id", "product_category_name_english"]
    ]
    .merge(
        master[
            [
                "order_id",
                "delivery_performance",
                "delivery_delay_days",
                "review_score",
                "low_review"
            ]
        ],
        on="order_id",
        how="left"
    )
)

category_delivery_analysis = (
    category_delivery[
        category_delivery["product_category_name_english"].notna()
    ]
    .groupby("product_category_name_english")
    .agg(
        orders=("order_id", "nunique"),
        late_rate=(
            "delivery_performance",
            lambda x: (x == "Late").mean() * 100
        ),
        avg_delay_days=("delivery_delay_days", "mean"),
        avg_review_score=("review_score", "mean"),
        low_review_rate=("low_review", "mean")
    )
    .reset_index()
)

category_delivery_analysis["low_review_rate"] *= 100

category_delivery_analysis = (
    category_delivery_analysis[
        category_delivery_analysis["orders"] >= 100
    ]
    .sort_values("low_review_rate", ascending=False)
)

category_delivery_analysis.head(15)

# 13. Seller Performance and Risk Segmentation

Seller-level performance is analyzed to identify differences in delivery reliability and customer satisfaction.

Sellers with at least 50 orders are included in the risk segmentation to reduce the influence of very small samples.

The analytical risk levels are based on observed late-delivery and low-review rates and do not represent official Olist classifications.

In [ ]:
seller_analysis = (
    order_items
    .groupby("seller_id")
    .agg(
        orders=("order_id", "nunique"),
        items_sold=("order_item_id", "count"),
        total_revenue=("price", "sum"),
        avg_price=("price", "mean"),
        avg_freight=("freight_value", "mean")
    )
    .reset_index()
)

seller_orders = (
    order_items[["order_id", "seller_id"]]
    .drop_duplicates()
    .merge(
        master[
            [
                "order_id",
                "delivery_performance",
                "delivery_delay_days",
                "review_score",
                "low_review"
            ]
        ],
        on="order_id",
        how="left"
    )
)

seller_delivery = (
    seller_orders
    .groupby("seller_id")
    .agg(
        late_rate=(
            "delivery_performance",
            lambda x: (x == "Late").mean() * 100
        ),
        avg_delay_days=("delivery_delay_days", "mean"),
        avg_review_score=("review_score", "mean"),
        low_review_rate=("low_review", "mean")
    )
    .reset_index()
)

seller_delivery["low_review_rate"] *= 100

seller_analysis = seller_analysis.merge(
    seller_delivery,
    on="seller_id",
    how="left"
)

seller_analysis_filtered = (
    seller_analysis[
        seller_analysis["orders"] >= 50
    ]
    .sort_values("late_rate", ascending=False)
)

seller_analysis_filtered.head(15)

In [ ]:
seller_risk = seller_analysis_filtered.copy()

seller_risk["risk_level"] = np.select(
    [
        (seller_risk["late_rate"] >= 15) &
        (seller_risk["low_review_rate"] >= 30),

        (seller_risk["late_rate"] >= 10) |
        (seller_risk["low_review_rate"] >= 20)
    ],
    [
        "High Risk",
        "Medium Risk"
    ],
    default="Low Risk"
)

risk_summary = (
    seller_risk
    .groupby("risk_level")
    .agg(
        sellers=("seller_id", "nunique"),
        total_orders=("orders", "sum"),
        avg_late_rate=("late_rate", "mean"),
        avg_low_review_rate=("low_review_rate", "mean"),
        avg_review_score=("avg_review_score", "mean")
    )
    .reset_index()
)

risk_summary

In [ ]:
high_risk_sellers = (
    seller_risk[
        seller_risk["risk_level"] == "High Risk"
    ]
    .sort_values(
        ["low_review_rate", "late_rate"],
        ascending=False
    )
)

high_risk_sellers.head(15)

In [ ]:
risk_plot = (
    risk_summary
    .sort_values("avg_late_rate")
)

plt.figure(figsize=(9, 5))

plt.bar(
    risk_plot["risk_level"],
    risk_plot["avg_late_rate"]
)

plt.title("Seller Risk Segmentation by Average Late Delivery Rate")
plt.xlabel("Risk Level")
plt.ylabel("Average Late Delivery Rate (%)")

plt.tight_layout()
plt.show()

### Key finding

Seller performance shows meaningful variation. High-risk sellers have substantially higher late-delivery and low-review rates than low-risk sellers, making seller monitoring a useful secondary operational priority.

# 14. Final Root Cause Matrix

The findings are consolidated into a root-cause matrix to compare the relative strength of different factors associated with customer dissatisfaction.

The ranking is based on the strength and consistency of observed evidence rather than a causal model.

In [ ]:
root_cause_matrix = pd.DataFrame({
    "factor": [
        "Delivery Reliability",
        "Delivery Delay Severity",
        "Seller Performance",
        "Product Category",
        "Freight Cost",
        "Payment Behavior"
    ],
    "evidence": [
        "Late vs On-Time low-review rate",
        "Low-review rate across delay buckets",
        "High/Medium/Low seller risk groups",
        "Category-level low-review rates",
        "Freight-ratio quintiles",
        "Payment type and installment analysis"
    ],
    "strength": [
        "Very Strong",
        "Very Strong",
        "Strong",
        "Moderate",
        "Weak",
        "Weak"
    ],
    "key_finding": [
        "54.56% low reviews for late orders vs 9.46% for on-time orders",
        "Low-review rate rises from 10.52% on-time to 78%+ for 8+ day delays",
        "High-risk sellers: 21.12% late rate and 41.36% low-review rate",
        "Several categories show elevated low-review rates",
        "Review score changes only slightly across freight-ratio groups",
        "Payment type/installments show limited review-score differences"
    ],
    "business_role": [
        "Primary",
        "Primary",
        "Secondary",
        "Secondary",
        "Weak/Secondary",
        "Limited"
    ]
})

root_cause_matrix

In [ ]:
strength_order = {
    "Very Strong": 4,
    "Strong": 3,
    "Moderate": 2,
    "Weak": 1
}

root_cause_plot = root_cause_matrix.copy()

root_cause_plot["strength_score"] = (
    root_cause_plot["strength"]
    .map(strength_order)
)

root_cause_plot = root_cause_plot.sort_values(
    "strength_score",
    ascending=True
)

plt.figure(figsize=(10, 6))

plt.barh(
    root_cause_plot["factor"],
    root_cause_plot["strength_score"]
)

plt.title("Relative Strength of Factors Associated with Low Reviews")
plt.xlabel("Evidence Strength")
plt.ylabel("Factor")

plt.tight_layout()
plt.show()

### Overall ranking

1. **Delivery Reliability — Very Strong — Primary**
2. **Delivery Delay Severity — Very Strong — Primary**
3. **Seller Performance — Strong — Secondary**
4. **Product Category — Moderate — Secondary**
5. **Freight Cost — Weak — Weak/Secondary**
6. **Payment Behavior — Weak — Limited**

# 15. Review Text Analysis

Customer review text is examined to complement the quantitative root-cause analysis.

The analysis focuses on low-score reviews and identifies commonly occurring terms and themes in review messages. This provides qualitative evidence about the issues customers mention in their own words.

Because review messages may be missing or may contain short and ambiguous text, the results are interpreted as supporting evidence rather than definitive causal evidence.

In [ ]:
# Select low-score reviews with available messages

low_reviews_text = order_reviews[
    (order_reviews["review_score"] <= 2) &
    (order_reviews["review_comment_message"].notna())
].copy()

print("Total low-score reviews:",
      (order_reviews["review_score"] <= 2).sum())

print("Low-score reviews with text:",
      len(low_reviews_text))

low_reviews_text[
    ["review_score", "review_comment_message"]
].head(10)

In [ ]:
# Basic text cleaning

low_reviews_text["review_text_clean"] = (
    low_reviews_text["review_comment_message"]
    .astype(str)
    .str.lower()
    .str.replace(r"[^a-zA-ZÀ-ÿ\s]", " ", regex=True)
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)

low_reviews_text[
    ["review_score", "review_text_clean"]
].head(10)

In [ ]:
# Check review text length

low_reviews_text["text_length"] = (
    low_reviews_text["review_comment_message"]
    .astype(str)
    .str.len()
)

low_reviews_text["text_length"].describe()

In [ ]:
# Common complaint keywords in low-score reviews

keywords = {
    "Delivery": [
        "entrega", "entregue", "atras", "atraso",
        "prazo", "chegar", "chegou", "demorou"
    ],
    "Product Quality": [
        "produto", "qualidade", "defeito",
        "quebrado", "ruim", "inferior"
    ],
    "Refund / Return": [
        "reembolso", "devolução", "devolver",
        "estorno", "dinheiro"
    ],
    "Missing / Incomplete": [
        "faltou", "faltando", "não recebi",
        "somente", "incompleto"
    ],
    "Seller / Support": [
        "vendedor", "resposta", "atendimento",
        "suporte", "contato"
    ]
}

keyword_results = []

texts = (
    low_reviews_text["review_comment_message"]
    .astype(str)
    .str.lower()
)

for theme, words in keywords.items():
    pattern = "|".join(words)
    matches = texts.str.contains(pattern, regex=True, na=False).sum()

    keyword_results.append({
        "theme": theme,
        "reviews_mentioning_theme": matches,
        "percentage_of_low_reviews_with_text":
            matches / len(low_reviews_text) * 100
    })

keyword_analysis = pd.DataFrame(keyword_results)

keyword_analysis = keyword_analysis.sort_values(
    "reviews_mentioning_theme",
    ascending=False
)

keyword_analysis

In [ ]:
plt.figure(figsize=(10, 5))

plt.bar(
    keyword_analysis["theme"],
    keyword_analysis["reviews_mentioning_theme"]
)

plt.title("Common Complaint Themes in Low-Score Reviews")
plt.xlabel("Complaint Theme")
plt.ylabel("Number of Reviews Mentioning Theme")
plt.xticks(rotation=25)

plt.tight_layout()
plt.show()

### Review Text Findings

The review-text analysis provides qualitative support for the quantitative findings. Among 15,093 low-score reviews, 11,408 contain written review messages, providing additional evidence about customer concerns.

The keyword-based analysis indicates that customers mention issues related to delivery, product quality, refunds or returns, missing or incomplete orders, and seller or customer support.

These findings support the broader analysis that delivery and fulfillment-related issues are important areas associated with customer dissatisfaction. However, keyword matching is used only as an exploratory thematic approach and should not be interpreted as a definitive classification of customer complaints.


# 16. Repeat Customer Analysis

Customer-level analysis uses `customer_unique_id` so that multiple orders from the same person can be identified.

Customers placing more than one order are classified as repeat customers; customers placing one order are classified as one-time customers.

In [ ]:
# Repeat Customer Analysis

customer_orders = (
    master.groupby("customer_unique_id")
    .agg(
        total_orders=("order_id", "nunique"),
        total_spent=("total_payment", "sum"),
        avg_review_score=("review_score", "mean")
    )
    .reset_index()
)

customer_orders["customer_type"] = np.where(
    customer_orders["total_orders"] > 1,
    "Repeat Customer",
    "One-Time Customer"
)

repeat_summary = (
    customer_orders
    .groupby("customer_type")
    .agg(
        customers=("customer_unique_id", "count"),
        avg_orders=("total_orders", "mean"),
        avg_spent=("total_spent", "mean"),
        avg_review_score=("avg_review_score", "mean")
    )
    .reset_index()
)

repeat_summary

In [ ]:
customer_type_counts = (
    customer_orders["customer_type"]
    .value_counts()
    .reset_index()
)

customer_type_counts.columns = ["customer_type", "customers"]

plt.figure(figsize=(8, 5))
plt.bar(
    customer_type_counts["customer_type"],
    customer_type_counts["customers"]
)

plt.title("One-Time vs Repeat Customers")
plt.xlabel("Customer Type")
plt.ylabel("Number of Customers")
plt.tight_layout()
plt.show()

## 20. Repeat Customer Analysis

Customer-level analysis shows that the marketplace is dominated by one-time customers, with 93,099 customers placing a single order, while 2,997 customers are classified as repeat customers.

Repeat customers place an average of 2.12 orders and have a higher average total spending of approximately 314.99 compared with 161.82 for one-time customers.

The average review scores are very similar between the two groups (4.10 for repeat customers versus 4.07 for one-time customers). Therefore, repeat purchasing appears to be more strongly associated with higher customer spending than with differences in review satisfaction.

This analysis suggests that repeat customers represent a relatively small customer group but contribute higher average spending.


# 17. Key Findings

1. **Delivery reliability is the strongest factor associated with customer dissatisfaction.** Late-delivered orders have a substantially higher low-review rate than on-time orders.
2. **Customer dissatisfaction increases sharply with delivery delay severity.** Low-review rates rise considerably as the number of days an order is delayed increases.
3. **Seller performance shows meaningful differences.** High-risk seller groups have considerably higher late-delivery and low-review rates than low-risk seller groups.
4. **Product categories show different satisfaction levels.** Several categories have higher-than-average low-review rates, although category effects are weaker than delivery-related factors.
5. **Freight cost has a relatively weak relationship with customer satisfaction.** Review scores remain relatively stable across freight-to-price groups.
6. **Payment behavior is more strongly related to order value than satisfaction.** Payment-method and installment differences show limited differences in review satisfaction.
7. **Review text provides qualitative support.** Customer comments contain themes related to delivery, product quality, refunds/returns, incomplete orders, and seller/support issues.
8. **Repeat customers spend more on average.** Repeat customers are a small group but have higher average total spending than one-time customers, while their average review score is similar.

# 18. Business Recommendations

### 1. Improve Delivery Reliability
Prioritize reducing late deliveries because delivery performance shows the strongest association with low customer reviews.

### 2. Monitor Delivery Delay Severity
Create operational alerts for orders approaching or exceeding their estimated delivery dates, with stronger intervention for severely delayed orders.

### 3. Identify and Support High-Risk Sellers
Regularly monitor sellers with elevated late-delivery and low-review rates and provide targeted operational support or performance reviews.

### 4. Investigate Underperforming Categories
Focus deeper quality and fulfillment analysis on high-volume categories with consistently elevated low-review rates.

### 5. Use Freight Cost as a Secondary Optimization Area
Freight cost should be monitored for efficiency, but the analysis suggests that it is not the primary explanation for customer dissatisfaction.

### 6. Maintain Payment Flexibility
Continue supporting dominant payment methods and installment options, while recognizing that payment behavior appears more relevant to order value than review satisfaction.

# 19. Limitations

* The analysis identifies associations between variables and customer reviews; it does not establish causal relationships.
* Some datasets contain missing values, particularly delivery-related timestamps for orders that were not delivered.
* Seller and category comparisons can be influenced by differences in order volume, so small samples should be interpreted cautiously.
* Geographic analysis is performed at the customer-state level and may not capture more detailed local logistics differences.
* Review scores provide an important measure of customer satisfaction, but they do not capture every aspect of the customer experience.
* Keyword matching in review text is exploratory and may miss context or classify a review under more than one theme.
* Very small seller, category, payment, and installment groups are not used for strong conclusions.

# 20. Conclusion

The analysis of the Olist Brazilian E-Commerce dataset shows that **delivery performance is the strongest observed factor associated with customer dissatisfaction**.

Both late-delivery frequency and delay severity show strong relationships with low review scores. Seller performance and product category provide additional, but comparatively weaker, signals of dissatisfaction.

Freight cost and payment behavior show limited relationships with review scores, although payment behavior is associated with order value. Review-text analysis provides qualitative support for delivery and fulfillment-related concerns, while repeat-customer analysis indicates that repeat customers have higher average spending with similar review satisfaction.

Overall, the findings suggest that improving delivery reliability, monitoring severe delays, and identifying high-risk sellers should be key operational priorities for improving customer satisfaction.

# 21. Power BI Export

The master dataset can be exported as a CSV for Power BI. This keeps the dashboard at the order-level grain and avoids accidental duplication from the raw item/payment tables.

In [ ]:
# Export the order-level master dataset for Power BI
powerbi_path = "/content/olist_master_for_powerbi.csv"
master.to_csv(powerbi_path, index=False)

print(f"Power BI dataset saved to: {powerbi_path}")
print(f"Shape: {master.shape[0]:,} rows × {master.shape[1]} columns")